- Used road_info.h5 to add static road features to the traffic data.
- Matched road records to traffic nodes using their node IDs.
- Added available road attributes, such as road length, lane count, speed limit, and centrality. These values remain constant for each node across time.

In [ ]:
import shutil
import h5py
import numpy as np
from pathlib import Path

traffic_path = Path("/data/mmtd/fixed_sensor_weather.h5")
road_path = Path("/data/mmtd/road_info.h5")
output_path = Path("/data/mmtd/fixed_sensor_weather_road.h5")

def node_key(value):
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, bytes):
        value = value.decode("utf-8")
    return str(value).strip()

with h5py.File(traffic_path, "r") as traffic_file, \
     h5py.File(road_path, "r") as road_file:

    traffic = traffic_file["data/values"][:].astype(np.float32)
    traffic_nodes = traffic_file["metadata/nodes"][:]
    traffic_features = [
        x.decode() if isinstance(x, bytes) else str(x)
        for x in traffic_file["metadata/features"][:]
    ]

    road_values = road_file["data/values"][:].astype(np.float32)
    road_nodes = road_file["metadata/nodes"][:]
    road_features = [
        x.decode() if isinstance(x, bytes) else str(x)
        for x in road_file["metadata/features"][:]
    ]

# Find the road_info row for each traffic node
road_row_by_id = {node_key(node): i for i, node in enumerate(road_nodes)}
missing = [node for node in traffic_nodes
           if node_key(node) not in road_row_by_id]

print("Traffic nodes:", len(traffic_nodes))
print("Road-info nodes:", len(road_nodes))
print("Unmatched traffic nodes:", len(missing))

if missing:
    raise ValueError(
        f"{len(missing)} traffic nodes have no road_info match. "
        f"Examples: {missing[:5]}"
    )

row_indices = [road_row_by_id[node_key(node)] for node in traffic_nodes]
road_aligned = road_values[row_indices]  # traffic-node order

T, N, _ = traffic.shape
road_over_time = np.broadcast_to(
    road_aligned[None, :, :], (T, N, road_aligned.shape[1])
).copy()

combined = np.concatenate([traffic, road_over_time], axis=2)
new_features = traffic_features + [f"road_{name}" for name in road_features]

# Save a new HDF5 file, preserving other groups and metadata
shutil.copy2(traffic_path, output_path)

with h5py.File(output_path, "r+") as f:
    value_attrs = dict(f["data/values"].attrs)
    del f["data/values"]
    new_values = f["data"].create_dataset(
        "values", data=combined, compression="gzip", compression_opts=4
    )
    for key, value in value_attrs.items():
        new_values.attrs[key] = value

    del f["metadata/features"]
    f["metadata"].create_dataset(
        "features",
        data=np.asarray(new_features, dtype=h5py.string_dtype("utf-8")),
    )

print("Saved:", output_path)
print("New shape:", combined.shape)
print("Features:", new_features)
print("Set model in_dim to:", combined.shape[2])